# Wavelet denoising + CLAHE

See each step on 10 dark images (one per class): original → CLAHE only → wavelet denoise → wavelet + CLAHE.

Why denoise first: CLAHE stretches contrast in dark regions, so it also stretches the sensor and JPEG noise there. Wavelet thresholding (the lecture notes name the idea in one line, PDF p.146; BayesShrink itself is from Chang, Yu & Vetterli, IEEE TIP 2000) removes small wavelet coefficients, which are mostly noise, before CLAHE amplifies them.

Images come from **official train** only (the pool that will be cleaned), resized to short side 256 like `01_original`. This notebook is for looking only. Don't pick parameters by looking at test images.

In [ ]:
SEED = 0           # change to draw 10 other images
DARK_ONLY = True   # draw from images with mean grey < 80
CLAHE_CLIP, CLAHE_TILE = 2.0, 8          # same as pipeline_02.md
WAVELET = dict(method="BayesShrink", mode="soft", rescale_sigma=True)

In [ ]:
import os
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage.restoration import denoise_wavelet, estimate_sigma

# ---- paths (preset) -------------------------------------------------------
# project root = the folder that contains pipeline_preprocess/ (works from any sub-folder)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline_preprocess").is_dir())
DATA_DIR = Path(os.environ.get("FOOD_DATA_DIR", PROJECT_ROOT / "data"))
IMG_DIR = DATA_DIR / "raw_data" / "images"
TRAIN_TXT = next((DATA_DIR / "raw_data").glob("**/train.txt"))

CLASSES = ["bibimbap", "miso_soup", "mussels", "prime_rib", "edamame",
           "baby_back_ribs", "paella", "chicken_wings", "guacamole", "nachos"]

In [ ]:
# one image per class, official train only
train_keys = set(TRAIN_TXT.read_text().split())
bright = pd.read_csv(DATA_DIR / "dataset" / "meta" / "brightness.csv", dtype={"id": str})
bright = bright[(bright.label + "/" + bright.id).isin(train_keys)]
if DARK_ONLY:
    bright = bright[bright.mean_grey < 80]

rng = random.Random(SEED)
picked = [(c, rng.choice(sorted(bright[bright.label == c].id))) for c in CLASSES]
picked

In [ ]:
def resize_short(img, short=256):
    h, w = img.shape[:2]
    s = short / min(h, w)
    return img if s >= 1 else cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)


def clahe(rgb):
    # lightness only, as in the pipeline
    l, a, b = cv2.split(cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB))
    l = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(CLAHE_TILE, CLAHE_TILE)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)


def wavelet(rgb):
    den = denoise_wavelet(rgb / 255.0, channel_axis=-1, convert2ycbcr=True, **WAVELET)
    return (np.clip(den, 0, 1) * 255).round().astype(np.uint8)


STAGES = {
    "original": lambda x: x,
    "CLAHE only": clahe,
    "wavelet": wavelet,
    "wavelet + CLAHE": lambda x: clahe(wavelet(x)),
}
images = []
for cls, img_id in picked:
    rgb = resize_short(cv2.cvtColor(cv2.imread(str(IMG_DIR / cls / f"{img_id}.jpg")), cv2.COLOR_BGR2RGB))
    images.append({name: f(rgb) for name, f in STAGES.items()})

## Whole images
One row per image, one column per stage.

In [ ]:
fig, axes = plt.subplots(len(images), len(STAGES), figsize=(13, 3.2 * len(images)))
for row, ((cls, img_id), stages) in zip(axes, zip(picked, images)):
    for ax, (name, img) in zip(row, stages.items()):
        ax.imshow(img)
        ax.set_title(f"{cls} {img_id}\n{name}" if name == "original" else name, fontsize=10)
        ax.axis("off")
fig.tight_layout()
plt.show()

## Zoom on the darkest patch
Noise is hard to see at full size. Each row zooms to the darkest 64×64 patch of the original, where CLAHE amplifies noise the most.

In [ ]:
def darkest_patch(rgb, size=64):
    grey = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    mean = cv2.blur(grey, (size, size))[size // 2:-size // 2, size // 2:-size // 2]
    y, x = np.unravel_index(mean.argmin(), mean.shape)
    return y, x


fig, axes = plt.subplots(len(images), len(STAGES), figsize=(11, 2.8 * len(images)))
for row, ((cls, img_id), stages) in zip(axes, zip(picked, images)):
    y, x = darkest_patch(stages["original"])
    for ax, (name, img) in zip(row, stages.items()):
        ax.imshow(img[y:y + 64, x:x + 64], interpolation="nearest")
        ax.set_title(f"{cls}\n{name}" if name == "original" else name, fontsize=10)
        ax.axis("off")
fig.tight_layout()
plt.show()

## Noise level per stage
`estimate_sigma` estimates Gaussian noise σ from the finest wavelet band (0–255 scale, mean over RGB). Expect CLAHE to raise it and the wavelet step to lower it. This is an estimate: on JPEG images it also picks up fine texture.

In [ ]:
sigma = pd.DataFrame(
    [{name: estimate_sigma(img, channel_axis=-1, average_sigmas=True) for name, img in stages.items()}
     for stages in images],
    index=[f"{c}/{i}" for c, i in picked]).round(2)
sigma.loc["mean"] = sigma.mean().round(2)
sigma